## Limpando os dados

Aqui vamos ter diversas situações com dados com problemas e realizaremos diversos tratamentos de como lidar com esses dados e realizar a limpeza deles

    Aqui usamos o comando toPandas() para visualizar os dados porém normalmente se deve usar o show() ou display()

**Limpando dados Nulos**

Exitem duas formas mais comuns de lidar com dados nulos

Remove-las ou Padroniza-las

In [1]:
# Importando Bibliotecas para uso do Spark para usar dentro do VSCODE
import findspark
findspark.init()
from pyspark.sql import SparkSession # Import para inciar o spark
from pyspark.sql.types import * # Todos as configurações do SQL de tipagem no spark
from pyspark.sql.functions import * # Todas as funções de SQL do Spark
# Criação da sessão Spark
spark = SparkSession.builder.appName("PySpark-VSCode-App").getOrCreate()
print("PySpark está pronto para uso!")

PySpark está pronto para uso!


**Abrindo DataFrame de Treino, para realizar ações**

In [ ]:
# Realizado a leitura de um arquivo CSV para criar um Dataframe com Spark (DataFrame)
txt_treino = r'C:\Users\Cleydenilson\Documents\Scripts\01 - APRENDIZADOS\07_PySpark\Data\Arquivo_Treino.txt'
df = spark.read.csv(txt_treino, sep=',', header=True, inferSchema=True, multiLine=True)

In [3]:
df.toPandas()

,Nome,Departamento,Salário
0,Pedro,None,2000.0
1,Jhon,Vendas,3000.0
2,Anna,Marketing,4500.0
3,João,Vendas,NaN
4,Mike,Vendas,3500.0
5,Sara,Marketing,4000.0
6,Maria,None,2000.0
7,Julia,,2500.0
8,Eduarda,\n,3000.0
9,None,None,NaN


#### Removendo dados que são nulos do meu Dataframe

Com o comando dropna() é possível retirar todos as linhas que contenha algum dado nulo

    how='all' : Comando adicionado ao dropna() para remover apenas as linhas que estão totalmente vaizas

In [4]:
# Remover qualquer campo que contenha nulo "dropna()"
df.dropna().toPandas()

,Nome,Departamento,Salário
0,Jhon,Vendas,3000
1,Anna,Marketing,4500
2,Mike,Vendas,3500
3,Sara,Marketing,4000
4,Julia,,2500
5,Eduarda,\n,3000


In [5]:
# Remover apenas a linha que contém null em todas as colunas "dropna" com argumento "how='all'"
df.dropna(how='all').toPandas()

,Nome,Departamento,Salário
0,Pedro,None,2000.0
1,Jhon,Vendas,3000.0
2,Anna,Marketing,4500.0
3,João,Vendas,NaN
4,Mike,Vendas,3500.0
5,Sara,Marketing,4000.0
6,Maria,None,2000.0
7,Julia,,2500.0
8,Eduarda,\n,3000.0


In [6]:
# Remove as linhas que contém dados nulos apenas das colunas definida no argumento "subset"
df.dropna(subset=["Departamento", "Nome"]).toPandas()

,Nome,Departamento,Salário
0,Jhon,Vendas,3000.0
1,Anna,Marketing,4500.0
2,João,Vendas,NaN
3,Mike,Vendas,3500.0
4,Sara,Marketing,4000.0
5,Julia,,2500.0
6,Eduarda,\n,3000.0


#### Trocado dados nulos por outra informação

Usando o comando filna(), é possível filtrar os dados nulos de uma coluna ou de todo ou DataFrame, para a informação desejada

In [7]:
df.fillna('Não Mapeado').toPandas()

,Nome,Departamento,Salário
0,Pedro,Não Mapeado,2000.0
1,Jhon,Vendas,3000.0
2,Anna,Marketing,4500.0
3,João,Vendas,NaN
4,Mike,Vendas,3500.0
5,Sara,Marketing,4000.0
6,Maria,Não Mapeado,2000.0
7,Julia,,2500.0
8,Eduarda,\n,3000.0
9,Não Mapeado,Não Mapeado,NaN


In [8]:
df.fillna('Não Mapeado', subset='Departamento').toPandas()

,Nome,Departamento,Salário
0,Pedro,Não Mapeado,2000.0
1,Jhon,Vendas,3000.0
2,Anna,Marketing,4500.0
3,João,Vendas,NaN
4,Mike,Vendas,3500.0
5,Sara,Marketing,4000.0
6,Maria,Não Mapeado,2000.0
7,Julia,,2500.0
8,Eduarda,\n,3000.0
9,None,Não Mapeado,NaN


Agora vamos trocar os Salarios Nulos pela média salarial geral

In [9]:
# Vamos extrair os dados da média salarial geral

df.select(avg('Salário')).collect()

[Row(avg(Salário)=3062.5)]

Como pode ver ele o collect me permite coletar o resultado da média salarial, porém dentro de uma lista então vamos entra na lista da usando o fatiamento para selecionar apenas o valor da média

In [10]:
media = df.select(avg('Salário')).collect()[0][0]

In [11]:
df.fillna(media, subset='Salário').toPandas()

,Nome,Departamento,Salário
0,Pedro,None,2000
1,Jhon,Vendas,3000
2,Anna,Marketing,4500
3,João,Vendas,3062
4,Mike,Vendas,3500
5,Sara,Marketing,4000
6,Maria,None,2000
7,Julia,,2500
8,Eduarda,\n,3000
9,None,None,3062


**Dados não nulos dados vazios**

Como reparou no Dataframe anterior você identificou que nem tudo que está vazio foi removido ou tratado!!!

O campo onde contém o departamento da Julia, João e Eduarada por exemplo tem espaços em brancos, tabulações ou quebra de linhas, então vamos precisar tratar isso para só então conseguir remover ou subsitituir

Então vamos usar um conjunto de comandos para realizar a limpeza de dados 

    regexp_replace() = Comando que controla strings e substituir por dados desejados

    when() = Comando condicional retorna um valor caso True e outro caso false

    trim() = Comando realiza a retirada de dados como tabulações quebra de linhas e espaços em brancos a esquerda ou a direita.

In [12]:
df = df.withColumn("Departamento", # Definido a coluna que desejo que seja feita a alteração
                regexp_replace( 
                    col("Departamento"), # Definido qual coluna deve sobre substituição dinamica com regex
                        "[\n\r]", '') # Comando Regex (\n = quebra de linhas, \r = recuo de carro), comando \n\r também conhecido como (CRLF)
                            )

In [13]:
df.toPandas()

,Nome,Departamento,Salário
0,Pedro,None,2000.0
1,Jhon,Vendas,3000.0
2,Anna,Marketing,4500.0
3,João,Vendas,NaN
4,Mike,Vendas,3500.0
5,Sara,Marketing,4000.0
6,Maria,None,2000.0
7,Julia,,2500.0
8,Eduarda,,3000.0
9,None,None,NaN


Com os dados completamente com strings vazias usamos o trim em conjunto com When para realizar o tratamento e retorna dados nulos

In [14]:
df.withColumn("Departamento", 
    # 1. Condição: Verifica se a coluna, após ser limpa (trim), é igual a uma string vazia ("")
    when(
        trim(col("Departamento")) == lit(""),
        lit(None) # 2. Se TRUE, retorna NULL
    ).otherwise(
        # 3. Se FALSE, retorna a string limpa (trim)
        trim(col("Departamento"))
    )
).toPandas()

,Nome,Departamento,Salário
0,Pedro,None,2000.0
1,Jhon,Vendas,3000.0
2,Anna,Marketing,4500.0
3,João,Vendas,NaN
4,Mike,Vendas,3500.0
5,Sara,Marketing,4000.0
6,Maria,None,2000.0
7,Julia,None,2500.0
8,Eduarda,None,3000.0
9,None,None,NaN
